# Intermediate 01 — Agent Memory Security Lab

Build a secure long-term memory lifecycle for a Northwind support agent. Model output is a proposal; trusted application code owns scope, provenance, consent, retention, versions, deletion, and authority.

## Learning path

1. Separate conversation history, checkpoints, durable memory, and systems of record.
2. Admit one user-confirmed preference and read it as untrusted data.
3. Compare a naive global string store with the policy boundary.
4. Exercise poisoning, concurrency, replay, consent, expiry, and deletion.
5. Calculate safety, utility, failure-state, and trace metrics.
6. Inspect real OpenAI Agents SDK tools and a real `SQLiteSession` without a model or credential.

In [ ]:
from dataclasses import replace
from datetime import timedelta
import importlib.util
from pathlib import Path
import sys

COURSE = Path.cwd()
if not (COURSE / 'lab.py').exists():
    COURSE = COURSE / 'curriculum' / 'roadmap' / 'intermediate' / '01-agent-memory-security'
spec = importlib.util.spec_from_file_location('intermediate_01_notebook_lab', COURSE / 'lab.py')
assert spec and spec.loader
lab = importlib.util.module_from_spec(spec)
sys.modules.setdefault(spec.name, lab)
spec.loader.exec_module(lab)
print(lab.POLICY_VERSION)

## 1. Classify state before securing it

Working context is transient. Session history continues a conversation. Checkpoints resume execution. Episodic and semantic memory persist selected events or facts. Procedural memory changes behavior and belongs to application/release owners. Mutable business state remains in its system of record. This lab persists only narrow semantic and episodic shapes.

In [ ]:
service, sources = lab.build_system()
write = service.write(lab.actor('notebook:write'), lab.proposal(sources['confirm:email']), now=lab.NOW)
read = service.read(lab.actor('notebook:read'), 'preferred_contact_method', now=lab.NOW)
assert write.status is lab.WriteStatus.CREATED
assert read.status is lab.ReadStatus.READY
assert read.memory.untrusted_data and not read.memory.grants_authority
{'write': write.status.value, 'read': read.status.value, 'value': read.memory.value, 'authority': read.memory.grants_authority}

## 2. Why a global string store fails

A naive store accepts whatever text the agent extracts and retrieves it by a global key. It has no authenticated scope, exact source, type policy, consent, version, TTL, or deletion epoch.

In [ ]:
naive_store = {}
naive_store['preferred_contact_method'] = 'email'  # trusts an agent summary
cross_subject_value = naive_store['preferred_contact_method']  # another subject sees it
assert cross_subject_value == 'email'
cross_subject_value

In [ ]:
service, sources = lab.build_system()
wrong_tenant = service.write(lab.actor('attack:tenant'), lab.proposal(sources['south:email']), now=lab.NOW)
agent_summary = service.write(
    lab.actor('attack:summary'),
    lab.proposal(sources['summary:email'], proposal_id='proposal:summary'),
    now=lab.NOW,
)
restricted = service.write(
    lab.actor('attack:restricted'),
    lab.proposal(sources['secret:token'], proposal_id='proposal:secret'),
    now=lab.NOW,
)
assert [item.status for item in (wrong_tenant, agent_summary, restricted)] == [lab.WriteStatus.DENY] * 3
[(item.reason, item.receipt.proposal_digest[:10]) for item in (wrong_tenant, agent_summary, restricted)]

## 3. Retries are not updates

An exact proposal retry is idempotent. The same proposal ID with different content is a collision. A correction names the current version so concurrent stale writers cannot overwrite it.

In [ ]:
service, sources = lab.build_system()
first_proposal = lab.proposal(sources['confirm:email'])
first = service.write(lab.actor('version:first'), first_proposal, now=lab.NOW)
retry = service.write(lab.actor('version:first'), first_proposal, now=lab.NOW)
stale_update = service.write(
    lab.actor('version:stale'),
    lab.proposal(sources['confirm:sms'], proposal_id='proposal:update', value='sms', expected_version=0),
    now=lab.NOW,
)
good_update = service.write(
    lab.actor('version:good'),
    lab.proposal(sources['confirm:sms'], proposal_id='proposal:update-2', value='sms', expected_version=1),
    now=lab.NOW + timedelta(minutes=1),
)
assert retry.status is lab.WriteStatus.IDEMPOTENT
assert stale_update.status is lab.WriteStatus.CONFLICT
assert good_update.status is lab.WriteStatus.UPDATED
[first.status.value, retry.status.value, stale_update.status.value, good_update.status.value]

## 4. Recheck consent and lifecycle on read

Write-time admission is not a permanent grant. Semantic memory becomes unavailable as soon as current consent is revoked.

In [ ]:
service, sources = lab.build_system()
service.write(lab.actor('consent:write'), lab.proposal(sources['confirm:email']), now=lab.NOW)
consent = service.consents.records['consent:personalization']
service.consents.records[consent.consent_id] = replace(consent, active=False)
revoked_read = service.read(lab.actor('consent:read'), 'preferred_contact_method', now=lab.NOW)
assert (revoked_read.status, revoked_read.reason) == (lab.ReadStatus.EMPTY, 'consent-revoked')
revoked_read

## 5. Deletion must prevent resurrection

Erasing a row is insufficient when delayed jobs and retries exist. Subject deletion advances a durable epoch; older writes can no longer repopulate the subject.

In [ ]:
service, sources = lab.build_system()
old_actor = lab.actor('delete:old')
old_proposal = lab.proposal(sources['confirm:email'])
service.write(old_actor, old_proposal, now=lab.NOW)
deleted = service.delete_subject(lab.actor('delete:now', purpose='privacy-delete'), now=lab.NOW + timedelta(minutes=1))
late = service.write(old_actor, replace(old_proposal, proposal_id='proposal:late'), now=lab.NOW + timedelta(minutes=2))
current_read = service.read(lab.actor('delete:read', epoch=1), 'preferred_contact_method', now=lab.NOW)
assert deleted.new_subject_epoch == 1
assert (late.status, late.reason) == (lab.WriteStatus.DENY, 'subject-epoch')
assert current_read.status is lab.ReadStatus.EMPTY
{'deleted': deleted.deleted_count, 'new_epoch': deleted.new_subject_epoch, 'late_retry': late.reason}

In [ ]:
report, cases = lab.evaluate_controls()
assert (report.cases, report.valid_cases, report.attack_cases, report.failure_cases) == (14, 4, 8, 2)
assert report.valid_task_success_rate == 1.0
assert report.attack_effect_rate == 0.0
assert report.baseline_attack_acceptance_rate == 1.0
report

## 6. Inspect the real SDK boundary

The write tool exposes only an allowed preference value. Identity, scope, source, consent, retention, versions, epoch, and idempotency remain in application context. The SQLite session demonstrates conversation continuity, not long-term memory governance.

In [ ]:
sdk_spec = importlib.util.spec_from_file_location('intermediate_01_notebook_sdk', COURSE / 'sdk_adapter.py')
assert sdk_spec and sdk_spec.loader
sdk = importlib.util.module_from_spec(sdk_spec)
sys.modules.setdefault(sdk_spec.name, sdk)
sdk_spec.loader.exec_module(sdk)
proof = await sdk.credential_free_demo_async()
assert proof['write_tool']['schema_fields'] == ['value']
assert proof['write_tool']['trusted_fields_excluded']
assert proof['session_is_conversation_history_not_memory_policy']
proof

## Production challenge

Add a strict `locale` preference, then implement revocation across a secondary index. Prove authorization filters run before ranking, stale concurrent updates conflict, and delayed epoch-0 ingestion cannot resurrect an epoch-1 subject. Document which guarantees your selected memory SDK provides and which remain application-owned.